# TextCNN Dropout 消融

研究问题：在当前基础 TextCNN 中，训练时使用 Dropout=0.5 与不使用 Dropout（0）相比，如何影响验证损失、F1 和训练过程？

控制训练／验证划分、训练词表、前 512 个词项与末尾填充规则、128 维词向量、64 个宽度 3 的卷积核、Adam（lr=0.001）、batch_size=32、最多 20 轮和耐心值 4。各次运行按最低验证损失保存最佳模型。

使用种子 42、123 两组配对实验；每个种子下两种配置均从头初始化，核对初始参数指纹和打乱顺序生成器种子相同。Dropout 所导致的后续随机计算差异是本次干预的一部分。设置确定性计算并记录环境，跨设备仍不保证完全一致。

这是已查看过验证结果和测试结果之后的补充探索实验，只使用训练集与验证集，不据此修改之前的测试结论。两种子仅用于初步观察波动。


In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# 加载数据和实验依赖
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.paths import CHECKPOINTS_DIR, CLEANED_DIR, TABLES_DIR
from sentiment_analysis.sequence_data import build_vocabulary, SentimentDataset, TOKEN_PATTERN
from sentiment_analysis.textcnn import TextCNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_data = pd.read_csv(CLEANED_DIR / "train.csv")
validation_data = pd.read_csv(CLEANED_DIR / "val.csv")

max_words = 20_000
max_length = 512
batch_size = 32
max_epochs = 20
patience = 4
learning_rate = 0.001

vocabulary = build_vocabulary(train_data["text"], max_words=max_words)
train_dataset = SentimentDataset(train_data, vocabulary, max_length)
validation_dataset = SentimentDataset(validation_data, vocabulary, max_length)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {device}")
print(f"Train: {len(train_dataset):,} | Validation: {len(validation_dataset):,}")
print(f"Vocabulary: {len(vocabulary):,}")

import hashlib
import platform
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)


In [ ]:
# 训练与验证
def train_one_epoch(model, data_loader, optimizer, loss_function):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for token_ids, lengths, labels in data_loader:
        token_ids = token_ids.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(token_ids, lengths)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate_model(model, data_loader, loss_function):
    model.eval()
    total_loss = 0.0
    total = 0
    y_true = []
    y_prediction = []

    with torch.no_grad():
        for token_ids, lengths, labels in data_loader:
            token_ids = token_ids.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            logits = model(token_ids, lengths)
            loss = loss_function(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total += labels.size(0)
            y_true.extend(labels.cpu().tolist())
            y_prediction.extend(logits.argmax(dim=1).cpu().tolist())

    return total_loss / total, classification_metrics(y_true, y_prediction)


In [ ]:
# 固定实验协议；保存时不覆盖已经存在的协议
import json
from sentiment_analysis.paths import ROOT
protocol_path = ROOT / "configs" / "textcnn_dropout_ablation.json"
input_paths = {
    "train": CLEANED_DIR / "train.csv",
    "validation": CLEANED_DIR / "val.csv",
    "model_code": ROOT / "src/sentiment_analysis/textcnn.py",
    "sequence_code": ROOT / "src/sentiment_analysis/sequence_data.py",
}
protocol = {
    "seeds": [42, 123], "dropout_values": [0.5, 0.0],
    "kernel_size": 3, "num_filters": 64, "embedding_dim": 128,
    "max_words": max_words, "max_length": max_length, "batch_size": batch_size,
    "optimizer": "Adam", "learning_rate": learning_rate,
    "max_epochs": max_epochs, "patience": patience,
    "selection": "minimum validation cross-entropy loss",
    "deterministic_algorithms": True,
    "python": platform.python_version(), "torch": str(torch.__version__),
    "cuda": torch.version.cuda,
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "inputs": {name: {"path": str(path.relative_to(ROOT)), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
               for name, path in input_paths.items()},
}
protocol_text = json.dumps(protocol, ensure_ascii=False, indent=2) + "\n"
protocol_path.parent.mkdir(parents=True, exist_ok=True)
if protocol_path.exists() and protocol_path.read_text(encoding="utf-8") != protocol_text:
    raise ValueError("A different Dropout ablation protocol already exists.")
protocol_path.write_text(protocol_text, encoding="utf-8")
print("Ablation protocol fixed.")


In [ ]:
# 每种配置从头训练，并保存最低验证损失对应的模型
initial_hashes = {}

def run_experiment(dropout, random_seed):
    torch.manual_seed(random_seed)
    generator = torch.Generator().manual_seed(random_seed)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=generator)

    config = {
        "vocabulary_size": len(vocabulary), "embedding_dim": 128,
        "num_filters": 64, "kernel_size": 3,
        "padding_index": vocabulary["<PAD>"], "dropout": dropout,
    }
    model = TextCNN(**config).to(device)
    digest = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        digest.update(name.encode())
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    initial_hash = digest.hexdigest()
    if random_seed in initial_hashes and initial_hashes[random_seed] != initial_hash:
        raise ValueError("Paired configurations have different initial weights.")
    initial_hashes[random_seed] = initial_hash

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    tag = f"textcnn_dropout{int(dropout * 100)}_seed{random_seed}"
    checkpoint_path = CHECKPOINTS_DIR / (tag + "_best.pt")
    if checkpoint_path.exists():
        raise FileExistsError(f"Existing ablation checkpoint: {checkpoint_path}")
    best_loss = float("inf")
    waiting = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, loss_function)
        validation_loss, metrics = evaluate_model(model, validation_loader, loss_function)
        history.append({
            "SEED": random_seed, "DROPOUT": dropout, "EPOCH": epoch,
            "TRAIN LOSS": train_loss, "TRAIN ACCURACY": train_accuracy,
            "VALIDATION LOSS": validation_loss, "VALIDATION ACCURACY": metrics["ACCURACY"],
            "VALIDATION F1": metrics["F1"],
        })
        print(f"Seed {random_seed} | Dropout {dropout} | Epoch {epoch} | "
              f"Train loss {train_loss:.4f} | Val loss {validation_loss:.4f} | F1 {metrics['F1']:.4f}",
              flush=True)

        if validation_loss < best_loss:
            best_loss = validation_loss
            waiting = 0
            best_result = {
                "SEED": random_seed, "DROPOUT": dropout, "BEST EPOCH": epoch,
                "VALIDATION LOSS": validation_loss, **metrics,
                "INITIAL WEIGHTS SHA256": initial_hash,
            }
            torch.save({
                "model_state_dict": model.state_dict(), "model_config": config,
                "vocabulary": vocabulary, "max_length": max_length,
                "token_pattern": TOKEN_PATTERN.pattern, "lowercase": True,
                "random_seed": random_seed, "epoch": epoch,
                "validation_loss": validation_loss, "initial_weights_sha256": initial_hash,
                "ablation_protocol_sha256": hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
            }, checkpoint_path)
        else:
            waiting += 1

        pd.DataFrame(history).to_csv(TABLES_DIR / (tag + "_history.csv"), index=False)
        if waiting >= patience:
            print("Early stopping.", flush=True)
            break

    best_result["STOPPED EPOCH"] = epoch
    return best_result


In [ ]:
# 四次运行：每个种子内只改变 Dropout
results = []
for random_seed in [42, 123]:
    for dropout in [0.5, 0.0]:
        result = run_experiment(dropout, random_seed)
        results.append(result)
        pd.DataFrame(results).to_csv(TABLES_DIR / "textcnn_dropout_ablation_results.csv", index=False)

ablation_results = pd.DataFrame(results)
display(ablation_results.round(4))


In [ ]:
# 配对比较：正差值表示 Dropout=0.5 的值更高
rows = []
for seed, data in ablation_results.groupby("SEED"):
    with_dropout = data[data["DROPOUT"] == 0.5].iloc[0]
    without_dropout = data[data["DROPOUT"] == 0].iloc[0]
    rows.append({
        "SEED": seed,
        "F1 WITH DROPOUT": with_dropout["F1"],
        "F1 WITHOUT DROPOUT": without_dropout["F1"],
        "F1 DIFFERENCE": with_dropout["F1"] - without_dropout["F1"],
        "LOSS WITH DROPOUT": with_dropout["VALIDATION LOSS"],
        "LOSS WITHOUT DROPOUT": without_dropout["VALIDATION LOSS"],
        "LOSS DIFFERENCE": with_dropout["VALIDATION LOSS"] - without_dropout["VALIDATION LOSS"],
    })
paired_comparison = pd.DataFrame(rows)
display(paired_comparison.round(4))
paired_comparison.to_csv(TABLES_DIR / "textcnn_dropout_ablation_paired.csv", index=False)


In [ ]:
# 绘制每个种子下的训练和验证损失
import matplotlib.pyplot as plt
from sentiment_analysis.paths import FIGURES_DIR

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, seed in zip(axes, [42, 123]):
    for dropout, color in [(0.5, "tab:blue"), (0.0, "tab:orange")]:
        tag = f"textcnn_dropout{int(dropout*100)}_seed{seed}"
        history = pd.read_csv(TABLES_DIR / (tag + "_history.csv"))
        axis.plot(history["EPOCH"], history["TRAIN LOSS"], color=color, linestyle="--",
                  label=f"Train, dropout={dropout}")
        axis.plot(history["EPOCH"], history["VALIDATION LOSS"], color=color,
                  label=f"Validation, dropout={dropout}")
    axis.set_title(f"Seed {seed}")
    axis.set_xlabel("Epoch")
    axis.set_ylabel("Cross-entropy loss")
    axis.legend(fontsize=8)
figure.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure.savefig(FIGURES_DIR / "textcnn_dropout_ablation_curves.png", dpi=150)
plt.show()


## 结果与限制

按预先固定的最低验证损失选择轮次后，Dropout=0.5 在两个配对种子下的验证损失均较低，F1 分别比 Dropout=0 高约 0.0023 和 0.0057。无 Dropout 的最低验证损失均出现在第 2 轮，有 Dropout 时为第 6 轮和第 5 轮。

无 Dropout 的训练损失下降较快，验证损失更早回升；这与更早过拟合的现象一致。但训练损失是在 train 模式下记录的，有 Dropout 时存在随机屏蔽，不能把两条训练损失的差距直接解释为泛化差距。

结论依赖模型选择规则。例如种子 42 的无 Dropout 模型在后续轮次出现过更高的 F1，但其验证损失已高于第 2 轮；本实验未改为按 F1 选择模型。不能把当前结果解释为 Dropout 对所有指标、所有轮次都更好。

同一种子内的初始权重指纹一致，四个最佳检查点均重新加载核对了验证指标。两个种子、同一已反复使用的验证集只提供探索性证据；本次实验没有使用测试集，也没有替换之前测试协议中的模型。
